In [8]:
import os
import sys
import time
import json
import pickle
import random
from datetime import datetime
from collections import Counter
from typing import List, Tuple

import numpy as np
import pandas as pd
import pytz
from tqdm import tqdm
from diskcache import Cache

import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence


def get_config(dataset: str) -> dict:
    if dataset == "mall":
        return {
            "dataset_name": "mall",
            "dataset_cache_dir": "./mall/cache",
            "data_chunk_keys": "data_chunk_keys",
            "chunk": "chunk",
            "result_dir": "./result",
            "batch_size": 512,
            "lr": 1e-4,
            "epoch": 50,
            "seed": 0,
            "word_embedding_size": 1024,
            "metric_feature_size": 37,
            "num_classes": 16,   # 这里按你的实际数据集改
        }
    elif dataset == "train-ticket":
        return {
            "dataset_name": "train-ticket",
            "dataset_cache_dir": "./train-ticket/cache",
            "data_chunk_keys": "data_chunk_keys",
            "chunk": "chunk",
            "result_dir": "./result",
            "batch_size": 512,
            "lr": 1e-4,
            "epoch": 50,
            "seed": 0,
            "word_embedding_size": 1024,
            "metric_feature_size": 37,
            "num_classes": 16,   # 这里按你的实际数据集改
        }
    else:
        raise ValueError("dataset must be 'mall' or 'train-ticket'")






In [9]:
import torch
from torch import nn
import math
from model.utils import GAT, GRUClassifier, CrossAttention, LSTMModel
from torch.nn.utils.rnn import pack_padded_sequence
import torch.nn.functional as F


class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()


class ConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_sizes, dilation=2):
        super(ConvNet, self).__init__()
        layers = []
        for i in range(len(kernel_sizes)):
            dilation_size = dilation ** i
            kernel_size = kernel_sizes[i]
            padding = (kernel_size - 1) * dilation_size
            in_channels = num_inputs if i == 0 else num_channels[i - 1]
            out_channels = num_channels[i]
            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size, padding=padding),
                nn.BatchNorm1d(out_channels), nn.ReLU(), Chomp1d(padding)]

        self.network = nn.Sequential(*layers)

        self.out_dim = num_channels[-1]

    def forward(self, x):
        x = x.permute(0, 2, 1).float()
        out = self.network(x)
        out = out.permute(0, 2, 1)
        return out


class SelfAttention(nn.Module):
    def __init__(self, input_size, seq_len):
        super(SelfAttention, self).__init__()
        self.atten_w = nn.Parameter(torch.randn(seq_len, input_size, 1))
        self.atten_bias = nn.Parameter(torch.randn(seq_len, 1, 1))
        self.glorot(self.atten_w)
        self.atten_bias.data.fill_(0)

    def forward(self, x):
        input_tensor = x.transpose(1, 0)  # w x b x h
        input_tensor = (torch.bmm(input_tensor, self.atten_w) + self.atten_bias)  # w x b x out
        input_tensor = input_tensor.transpose(1, 0)
        atten_weight = input_tensor.tanh()
        weighted_sum = torch.bmm(atten_weight.transpose(1, 2), x).squeeze()
        return weighted_sum

    def glorot(self, tensor):
        if tensor is not None:
            stdv = math.sqrt(7.0 / (tensor.size(-2) + tensor.size(-1)))
            tensor.data.uniform_(-stdv, stdv)


class GaussianAttention(nn.Module):
    def __init__(self, input_size, seq_len):
        super(GaussianAttention, self).__init__()
        self.atten_w = nn.Parameter(torch.randn(seq_len, input_size, 1))
        self.atten_bias = nn.Parameter(torch.randn(seq_len, 1, 1))
        self.glorot(self.atten_w)
        self.atten_bias.data.fill_(0)
        self.seq_len = seq_len
        self.batch_norm = nn.BatchNorm1d(input_size)

        # Gaussian parameters
        self.mu = torch.tensor(seq_len // 2, dtype=torch.float)  # initialized at the center
        self.sigma = torch.tensor(5.0, dtype=torch.float)  # initialized with 5

    def forward(self, x):
        # x: [batch_size, window_size, input_size]
        batch_size = x.size(0)
        input_size = x.size(2)

        # Calculate Gaussian weights
        positions = torch.arange(self.seq_len, dtype=torch.float).unsqueeze(1).to(x.device)  # Shape: (w, 1)
        gauss_weight = 1.0 / (math.sqrt(2 * math.pi) * self.sigma) * torch.exp(
            -0.5 * ((positions - self.mu) / self.sigma) ** 2).repeat(1, input_size)
        gauss_weight = gauss_weight.unsqueeze(0).repeat(batch_size, 1, 1)  # Normalize w*input

        x = x.transpose(1, 2)  # x: [batch_size, input_size, window_size]
        x = self.batch_norm(x)
        x = x.transpose(1, 2)  # x: [batch_size, window_size, input_size]
        x = x + gauss_weight

        input_tensor = x.transpose(1, 0)  # w x b x h
        input_tensor = (torch.bmm(input_tensor, self.atten_w) + self.atten_bias)  # w x (b x out)
        input_tensor = input_tensor.transpose(1, 0)  # w x b x out
        atten_weight = input_tensor.tanh()  #
        weighted_sum = torch.bmm(atten_weight.transpose(1, 2), x).squeeze()
        return weighted_sum

    def glorot(self, tensor):
        if tensor is not None:
            stdv = math.sqrt(6.0 / (tensor.size(-2) + tensor.size(-1)))
            tensor.data.uniform_(-stdv, stdv)


class MetricModel(nn.Module):
    def __init__(self, metric_num, metric_hiddens=[64], metric_kernel_sizes=[2], self_attn=True,
                 chunk_lenth=41):
        super(MetricModel, self).__init__()
        self.metric_num = metric_num
        self.out_dim = metric_hiddens[-1]
        in_dim = metric_num

        assert len(metric_hiddens) == len(metric_kernel_sizes)
        self.net = ConvNet(num_inputs=in_dim, num_channels=metric_hiddens, kernel_sizes=metric_kernel_sizes)

        self.self_attn = self_attn
        if self_attn:
            assert (chunk_lenth is not None)
            self.attn_layer = GaussianAttention(self.out_dim, chunk_lenth)

    def forward(self, x):
        assert x.shape[-1] == self.metric_num
        # 使用一维卷积神经网络（1D CNN）对不同维度的指标进行融合
        hidden_states = self.net(x) # [N, padding_time_length, num_metrics]-->(N, T, C)
        if self.self_attn:
            return self.attn_layer(hidden_states) #在时间序列上做attention,高斯编码突出trace采样点
        return hidden_states[:, -1, :]


class LogModel(nn.Module):
    def __init__(self, embedding_size, out_dim=64):
        super(LogModel, self).__init__()
        self.lstm = LSTMModel(embedding_size, out_dim)

    def forward(self, x, perm_idx):  # [bz, event_num]
        """
        Input:
            paras: mu with length of event_num
        """
        x = self.lstm(x) # 时间顺序建模 [N, L_max, D_text]-->[N, hidden]
        _, unperm_idx = perm_idx.sort(0)
        x = x[unperm_idx]
        return x


class TraceModel(nn.Module):
    def __init__(self, embedding_size, out_dim=64):
        super(TraceModel, self).__init__()
        self.encoder1 = nn.Linear(1, out_dim)
        self.encoder2 = nn.Linear(1, out_dim)
        self.encoder3 = GAT(embedding_size, out_dim)

        self.fc1 = nn.Linear(3 * out_dim, 3 * 128)
        self.fc2 = nn.Linear(3 * 128, 128)
        self.fc3 = nn.Linear(128, out_dim)

    def forward(self, x, edge_index):  # [bz, event_num]
        """
        Input:
            paras: mu with length of event_num
        """
        x1 = x[:, 0:1] # Duration (N, 1)
        x2 = x[:, 1:2] # ResultCode (N, 1)
        x3 = x[:, 2:] # RpcName embedding (N, Bert_hidden)
        x1 = self.encoder1(x1) # (N, 1)-->(N, hidden)
        x2 = self.encoder2(x2) # (N, 1)-->(N, hidden)
        x3 = self.encoder3(x3, edge_index)  # (N, Bert_hidden)-->(N, hidden)
        # 将全连接编码的duration和返回状态码,和GAT编码的接口服务名,用MLP拼接融合
        x = torch.cat((x1, x2, x3), dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


class base_model(nn.Module):
    def __init__(self, embedding_size, metric_feature_size, num_classes, gat_output_size=64):
        super(base_model, self).__init__()
        self.trace_feature_extractor = TraceModel(embedding_size, 64)
        self.log_feature_extractor = LogModel(embedding_size, out_dim=64)
        self.metric_feature_extractor = MetricModel(metric_feature_size, metric_hiddens=[64])

        self.trace_attention = CrossAttention(64)
        self.log_attention = CrossAttention(64)
        self.metric_attention = CrossAttention(64)

        self.fuse = nn.Linear(6 * 64, 128)
        self.activate = nn.GLU()

        self.classifier = GRUClassifier(gat_output_size, num_classes)

    def forward(self, trace_feature, log_feature, metric_feature, perm_idx, edge_index, node_length):
        # [N, 2 + D_text]-->(N, hidden)
        trace_x = self.trace_feature_extractor(trace_feature, edge_index)
        # [N, L_max, D_text]-->(N, hidden)
        log_x = self.log_feature_extractor(log_feature, perm_idx)
        # [N, padding_time_length, num_metrics]-->(N, hidden)
        metric_x = self.metric_feature_extractor(metric_feature[:, :, :37])

        x1 = self.trace_attention(trace_x, log_x)
        x2 = self.trace_attention(trace_x, metric_x)
        x3 = self.log_attention(log_x, trace_x)
        x4 = self.log_attention(log_x, metric_x)
        x5 = self.metric_attention(metric_x, trace_x)
        x6 = self.metric_attention(metric_x, log_x)

        fusion_feature = self.activate(self.fuse(torch.cat((x1, x2, x3, x4, x5, x6), dim=-1)))

        split_node_tensor = torch.split(fusion_feature, node_length)
        # padding
        sequence_lengths = [len(seq) for seq in split_node_tensor]
        sequence_lengths, perm_idx = torch.sort(torch.LongTensor(sequence_lengths), descending=True)
        split_node_tensor = [split_node_tensor[i] for i in perm_idx]
        max_length = max(sequence_lengths)
        padded_sequences = torch.zeros(len(split_node_tensor), max_length, split_node_tensor[0].shape[1])

        for i, seq in enumerate(split_node_tensor):
            end = sequence_lengths[i]
            padded_sequences[i, :end, :] = seq
        log_packed_input = pack_padded_sequence(padded_sequences, sequence_lengths, batch_first=True).cuda()
        output = self.classifier(log_packed_input)
        _, unperm_idx = perm_idx.sort(0)
        output = output[unperm_idx]

        return output


In [10]:
class Trainer:
    """
    作用：
    1. 从 cache 读取构图后的 data_list
    2. 划分训练 / 测试集
    3. 构建 batch 的三模态输入
    4. 训练 base_model
    5. 测试并保存结果
    """

    def __init__(self, config: dict):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = None
        self.train_loader = None
        self.test_loader = None
        self.X_train = None
        self.X_test = None

    # =========================
    # 基础工具函数（来自 utils.py 思路）
    # =========================
    @staticmethod
    def set_seed(seed=0):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.use_deterministic_algorithms(True)

    @staticmethod
    def print_label(y_label, prefix):
        print(prefix)
        counter = Counter(y_label)
        sorted_counter = sorted(counter.items(), key=lambda x: x[0])
        for element, count in sorted_counter:
            print(f"Class {element}: {count}")
        print(f"Total: {len(y_label)}")

    @staticmethod
    def stratified_train_test_split(X, y, test_size=0.3):
        unique_labels = np.unique(y)
        X_train, X_test, y_train, y_test = [], [], [], []

        for label in unique_labels:
            X_label = [X[i] for i in range(len(X)) if y[i] == label]
            y_label = [y[i] for i in range(len(X)) if y[i] == label]

            split_point = int(len(X_label) * (1 - test_size))

            X_train.append(X_label[:split_point])
            X_test.append(X_label[split_point:])
            y_train.append(y_label[:split_point])
            y_test.append(y_label[split_point:])

        X_train = [item for a_list in X_train for item in a_list]
        X_test = [item for a_list in X_test for item in a_list]
        y_train = [item for a_list in y_train for item in a_list]
        y_test = [item for a_list in y_test for item in a_list]

        return X_train, X_test, y_train, y_test

    def load_cached_data(self) -> List:
        """
        从 diskcache 中恢复图样本 data_list
        对应 utils.py 里的 get_data() 逻辑
        """
        cache = Cache(self.config["dataset_cache_dir"], size_limit=int(9 * 1e10))
        data_list = []

        chunk_keys = cache.get(self.config["data_chunk_keys"])
        if chunk_keys is None:
            raise ValueError("No cached graph data found. Please run graph building first.")

        for key in tqdm(chunk_keys, postfix="load cached graph data"):
            chunk = pickle.loads(cache.get(key))
            data_list.extend(chunk)

        return data_list

    def get_dataset(self) -> Tuple[List, List]:
        """
        对应 utils.py 里的 get_dataset()
        """
        data_list = self.load_cached_data()
        label_list = [data.label for data in data_list]

        # 原始源码是 7:3 划分
        X_train, X_test, _, _ = self.stratified_train_test_split(
            data_list, label_list, test_size=3 / (7 + 3)
        )

        return X_train, X_test

    # =========================
    # 模型与 DataLoader
    # =========================
    def build_model(self):
        self.model = base_model(
            embedding_size=self.config["word_embedding_size"],
            metric_feature_size=self.config["metric_feature_size"],
            num_classes=self.config["num_classes"],
        ).to(self.device)
        return self.model

    def build_dataloaders(self):
        self.X_train, self.X_test = self.get_dataset()

        y_train = [data.label for data in self.X_train]
        y_test = [data.label for data in self.X_test]

        self.print_label(y_train, "Training set")
        self.print_label(y_test, "Testing set")

        self.train_loader = DataLoader(
            self.X_train,
            batch_size=self.config["batch_size"],
            shuffle=True,
        )
        self.test_loader = DataLoader(
            self.X_test,
            batch_size=self.config["batch_size"],
            shuffle=True,
        )

        return self.train_loader, self.test_loader

    # =========================
    # batch -> 三模态输入
    # 对应 main.py 里的 get_mutimodal_feature()
    # =========================
    def get_multimodal_feature(self, batch_data):
        """
        输入：
            batch_data: PyG batch

        输出：
            batch_trace_feature
            log_packed_input
            batch_metrics_feature
            perm_idx
            edge_index
            node_length
        """
        # 每个 graph 的节点数
        node_length = [len(list_x) for list_x in batch_data.x]

        # 1. trace_feature
        # batch内所有图所有节点的trace特征 [N, 2 + embedding_size]
        batch_trace_feature = torch.stack(
            [x["trace_feature"] for list_x in batch_data.x for x in list_x]
        ).to(self.device)

        # 2. metrics_feature
        # batch内所有图所有节点的metric特征[N, padding_length, num_metrics] [总节点数, 时间长度, 指标数]
        batch_metrics_feature = torch.stack(
            [x["metrics_feature"][:39, ] for list_x in batch_data.x for x in list_x]
        ).permute(0, 2, 1).to(self.device)

        # 3. logs_feature：变长序列，需要 padding + pack
        # batch_logs_feature：长度为 N 的 list，里面每个元素都是一个节点的日志序列张量。[L_i, D_text]
        batch_logs_feature = [x["logs_feature"] for list_x in batch_data.x for x in list_x]
        # 长度为 N 的 list / tensor，每个值是一个节点的日志条数
        sequence_lengths = [len(seq) for seq in batch_logs_feature]

        # 把所有节点的日志序列 padding 到同长度 [N, L_max, D_text]
        sequence_lengths, perm_idx = torch.sort(
            torch.LongTensor(sequence_lengths), descending=True
        )
        batch_logs_feature = [batch_logs_feature[i] for i in perm_idx]

        max_length = max(sequence_lengths)
        padded_sequences = torch.zeros(
            len(batch_logs_feature),
            max_length,
            batch_logs_feature[0].shape[1],
        )

        for i, seq in enumerate(batch_logs_feature):
            end = sequence_lengths[i]
            padded_sequences[i, :end, :] = seq

        log_packed_input = pack_padded_sequence(
            padded_sequences,
            sequence_lengths,
            batch_first=True,
        ).to(self.device)

        # [2, E] batch中所有图合并后的总边数
        edge_index = batch_data.edge_index.to(self.device)

        return (
            batch_trace_feature,
            log_packed_input,
            batch_metrics_feature,
            perm_idx,
            edge_index,
            node_length,
        )

    # =========================
    # 评估
    # 对应 utils.py output_metrics()
    # =========================
    @staticmethod
    def output_metrics(result_df):
        from sklearn.metrics import precision_score, recall_score, f1_score

        try:
            y_pred = np.array([np.array(eval(item)) for item in list(result_df["y_pred"])])
        except Exception:
            y_pred = np.array([np.array(item) for item in list(result_df["y_pred"])])

        y_true = np.array(list(result_df["y_true"]))
        y_pred = np.array([item[0] for item in y_pred])

        macro_precision = precision_score(y_true, y_pred, average="macro")
        macro_recall = recall_score(y_true, y_pred, average="macro")
        macro_f1 = f1_score(y_true, y_pred, average="macro")

        print(
            f"Macro Precision: {macro_precision * 100:.2f}% "
            f"Recall: {macro_recall * 100:.2f}% "
            f"F1-Score: {macro_f1 * 100:.2f}%"
        )

        return {
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
        }

    def test(self):
        self.model.eval()
        y_pred = []
        y_true = []

        with torch.no_grad():
            for batch_data in self.test_loader:
                (
                    batch_trace_feature, # [N, 2 + embedding_size]
                    log_packed_input, # [N, L_max, D_text]
                    batch_metrics_feature, # [N, padding_time_length, num_metrics]
                    perm_idx, # [N], 按日志长度降序排序后，每个位置对应原来的哪个节点
                    edge_index, # [2, E]
                    node_length, # [N_1, N_2, ..., N_B] 每张图的节点数
                ) = self.get_multimodal_feature(batch_data)

                label = batch_data.label.to(self.device) # [B],图级标签

                output = self.model(
                    batch_trace_feature,
                    log_packed_input,
                    batch_metrics_feature,
                    perm_idx,
                    edge_index,
                    node_length,
                )

                y_pred += list(torch.argsort(output, dim=1, descending=True).cpu().numpy())
                y_true += list(label.cpu().numpy())

        result_df = pd.DataFrame(
            {
                "y_pred": [list(item) for item in y_pred],
                "y_true": y_true,
            }
        )

        metrics = self.output_metrics(result_df)
        return result_df, metrics

    # =========================
    # 保存结果
    # =========================
    def save_result(self, result_df):
        os.makedirs(self.config["result_dir"], exist_ok=True)

        script_name = os.path.basename(sys.argv[0]) if len(sys.argv) > 0 else "trainer"
        beijing_tz = pytz.timezone("Asia/Shanghai")
        beijing_time = datetime.now(beijing_tz)
        time_str = beijing_time.strftime("%m-%d_%H-%M")

        file_name = f"result_{os.path.splitext(script_name)[0]}_{time_str}.csv"
        save_result_file = os.path.join(self.config["result_dir"], file_name)

        print(f"save file to {save_result_file}")
        result_df.to_csv(save_result_file, index=False)

    # =========================
    # 训练
    # =========================
    def train(self):
        if self.model is None:
            self.build_model()
        if self.train_loader is None or self.test_loader is None:
            self.build_dataloaders()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(
            self.model.parameters(),
            lr=float(self.config["lr"]),
            weight_decay=5e-6,
        )

        average_loss_dict = {}

        epoch_iter = tqdm(range(self.config["epoch"]), desc="Training")
        for epoch in epoch_iter:
            self.model.train()
            total_loss = 0.0

            for batch_data in self.train_loader:
                (
                    batch_trace_feature,
                    log_packed_input,
                    batch_metrics_feature,
                    perm_idx,
                    edge_index,
                    node_length,
                ) = self.get_multimodal_feature(batch_data)

                label = batch_data.label.to(self.device)

                output = self.model(
                    batch_trace_feature,
                    log_packed_input,
                    batch_metrics_feature,
                    perm_idx,
                    edge_index,
                    node_length,
                )

                loss = criterion(output, label)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            average_loss = total_loss / len(self.train_loader)
            average_loss_dict[epoch] = average_loss
            epoch_iter.set_postfix({"avg_loss": average_loss})

        return average_loss_dict

    # =========================
    # 训练 + 测试 + 推理时间
    # =========================
    def run(self):
        self.set_seed(self.config["seed"])
        self.build_model()
        self.build_dataloaders()

        print("\nStart training...")
        loss_history = self.train()

        print("\nStart testing...")
        start_t = time.time()
        result_df, metrics = self.test()
        end_t = time.time()

        inference_time = (end_t - start_t) / len(self.X_test)
        print(f"Inference time per sample: {inference_time}")

        self.save_result(result_df)

        return {
            "loss_history": loss_history,
            "result_df": result_df,
            "metrics": metrics,
            "inference_time": inference_time,
        }


In [11]:
for dataset in ["mall", "train-ticket"]:   # "mall" 或 "train-ticket"

    config = get_config(dataset)
    trainer = Trainer(config)
    output = trainer.run()
    
    print("\nFinal metrics:")
    print(output["metrics"])

100%|██████████| 26/26 [03:20<00:00,  7.71s/it, load cached graph data]
/tmp/ipykernel_2212632/2481349334.py:120: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  self.train_loader = DataLoader(
/tmp/ipykernel_2212632/2481349334.py:125: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  self.test_loader = DataLoader(


Training set
Class 0: 5315
Class 1: 3551
Class 2: 5113
Class 3: 654
Class 4: 1348
Class 5: 2032
Class 6: 5625
Class 7: 2765
Class 8: 727
Class 9: 720
Class 10: 4555
Class 11: 2837
Total: 35242
Testing set
Class 0: 2279
Class 1: 1522
Class 2: 2192
Class 3: 281
Class 4: 578
Class 5: 872
Class 6: 2411
Class 7: 1186
Class 8: 312
Class 9: 309
Class 10: 1953
Class 11: 1217
Total: 15112

Start training...


Training: 100%|██████████| 50/50 [37:50<00:00, 45.41s/it, avg_loss=0.0522]



Start testing...
Macro Precision: 98.32% Recall: 98.12% F1-Score: 98.21%
Inference time per sample: 0.001136877080509434
save file to ./result/result_ipykernel_launcher_04-03_18-24.csv

Final metrics:
{'macro_precision': 0.9832439978702534, 'macro_recall': 0.9812276996688123, 'macro_f1': 0.9820909962463364}


100%|██████████| 2/2 [00:10<00:00,  5.17s/it, load cached graph data]
/tmp/ipykernel_2212632/2481349334.py:120: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  self.train_loader = DataLoader(
/tmp/ipykernel_2212632/2481349334.py:125: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  self.test_loader = DataLoader(


Training set
Class 0: 145
Class 1: 110
Class 2: 241
Class 3: 106
Class 4: 102
Class 5: 53
Class 6: 206
Class 7: 161
Class 8: 157
Class 9: 156
Class 10: 161
Class 11: 163
Total: 1761
Testing set
Class 0: 63
Class 1: 48
Class 2: 104
Class 3: 46
Class 4: 45
Class 5: 23
Class 6: 89
Class 7: 69
Class 8: 68
Class 9: 67
Class 10: 69
Class 11: 70
Total: 761

Start training...


Training: 100%|██████████| 50/50 [01:16<00:00,  1.53s/it, avg_loss=0.525]



Start testing...
Macro Precision: 89.48% Recall: 83.28% F1-Score: 82.87%
Inference time per sample: 0.0007165953928161701
save file to ./result/result_ipykernel_launcher_04-03_18-25.csv

Final metrics:
{'macro_precision': 0.8948366043424577, 'macro_recall': 0.8328189220564511, 'macro_f1': 0.8287491808218114}
